# Error Analysis

この notebook は、評価済み feature rows から false positives / false negatives を観察するための作業台です。

precision / recall は全体の成績を見るために使い、この notebook では「どの行を間違えたか」を見ます。実データではなく合成 fixture だけを使います。

## 1. feature rows を読み込む

feature row は、ある `user_id` をある `as_of_time` 時点で評価するための特徴量1行です。

ここでは `scoring_fn` が使う特徴量と、評価用の `label_value` が入った合成 CSV を読み込みます。

In [ ]:
from pathlib import Path
import sys

import pandas as pd

repo_candidates = [Path.cwd(), Path.cwd().parent]
repo_root = next(path for path in repo_candidates if (path / "src").exists())
sys.path.insert(0, str(repo_root / "src"))

from abuse_detection.evaluation import evaluate_feature_rows, load_feature_rows
from abuse_detection.error_analysis import (
    false_negatives,
    false_positives,
    score_bucket_summary,
)
from abuse_detection.scoring import scoring_fn

fixture_path = repo_root / "fixtures" / "feature_rows_sample.csv"
feature_rows = load_feature_rows(str(fixture_path))
feature_rows.head()

## 2. scoring_fn を適用する

`scoring_fn` は label を見ず、feature row だけから `risk_score` を返します。

error analysis は、そのあとに `risk_score` と `label_value` を見て誤判定を観察します。

In [ ]:
thresholds = list(range(0, 101, 10))
selected_threshold = 80

scored_rows, metrics = evaluate_feature_rows(
    feature_rows,
    thresholds=thresholds,
    score_fn=scoring_fn,
)

metrics

## 3. threshold=80 の全体成績を見る

まず precision / recall で全体像を見ます。

ただし、この表だけではどの user row を間違えたかは分かりません。

In [ ]:
metrics[metrics["threshold"] == selected_threshold]

## 4. score bucket で分布を見る

個別行を見る前に、score 帯ごとの分布を見ます。

`false_positive_count` が高 score bucket に多いなら、正常ユーザーを疑いすぎています。`false_negative_count` が低 score bucket に多いなら、abuse ユーザーを見逃しています。

In [ ]:
bucket_summary = score_bucket_summary(
    scored_rows,
    bucket_size=20,
    threshold=selected_threshold,
)

bucket_summary

## 5. false positives を読む

false positive は、実際は non-abuse なのに `risk_score >= threshold` になった行です。

これは誤検知なので、正常ユーザーを止めすぎていないかを見るために読みます。

In [ ]:
feature_columns = [
    "account_age_minutes",
    "contacts_24h",
    "messages_1h",
    "profile_updates_24h",
    "device_count_24h",
    "failed_login_count_24h",
    "login_country_changes_24h",
    "password_reset_24h",
    "recipient_block_rate_24h",
    "message_link_ratio_1h",
    "plan",
]

fps = false_positives(scored_rows, threshold=selected_threshold)
fps[["user_id", "label_value", "risk_score", *feature_columns]].sort_values(
    "risk_score",
    ascending=False,
)

## 6. false negatives を読む

false negative は、実際は abuse なのに `risk_score < threshold` になった行です。

これは見逃しなので、scoring_fn が拾えていない abuse pattern を探すために読みます。

In [ ]:
fns = false_negatives(scored_rows, threshold=selected_threshold)
fns[["user_id", "label_value", "risk_score", *feature_columns]].sort_values(
    "risk_score",
    ascending=False,
)

## 7. 誤判定 row の特徴量平均を見る

個別行を読むだけでなく、false positive / false negative の特徴量平均も見ます。

平均との差分を見ると、どの特徴量が誤判定に関係していそうか仮説を立てやすくなります。

In [ ]:
numeric_feature_columns = [
    "account_age_minutes",
    "contacts_24h",
    "messages_1h",
    "profile_updates_24h",
    "device_count_24h",
    "failed_login_count_24h",
    "login_country_changes_24h",
    "password_reset_24h",
    "recipient_block_rate_24h",
    "message_link_ratio_1h",
]

comparison_frames = []
for name, rows in {
    "all_rows": scored_rows,
    "false_positives": fps,
    "false_negatives": fns,
}.items():
    means = rows[numeric_feature_columns].mean().rename(name)
    comparison_frames.append(means)

feature_mean_comparison = pd.concat(comparison_frames, axis=1)
feature_mean_comparison["fp_minus_all"] = (
    feature_mean_comparison["false_positives"] - feature_mean_comparison["all_rows"]
)
feature_mean_comparison["fn_minus_all"] = (
    feature_mean_comparison["false_negatives"] - feature_mean_comparison["all_rows"]
)

feature_mean_comparison

## 8. 観察メモ

この notebook では、次の順番で error analysis を進めます。

1. precision / recall で全体を見る
2. score bucket で誤りの位置を見る
3. false positives を読んで誤検知 pattern を探す
4. false negatives を読んで見逃し pattern を探す
5. 特徴量平均を見て scoring_fn 改善候補を考える

次に試す候補:

* `selected_threshold` を 60 / 70 / 90 に変える
* `bucket_size` を 10 に変えて score 分布を細かく見る
* false positive に多い特徴量の加点を弱める
* false negative に多い特徴量の加点を強める
* `01_evaluate_scoring.ipynb` の conservative / recall_heavy と比較する